# Colab TTS Quality Gate — clean isolated environments

This notebook deliberately does **not** install Qwen and Chatterbox into the Colab kernel. The previous report showed `torch 2.11.0+cu128` mixed with package files and failed inside `torch._C._dynamo` (`skip_code`). That means the runtime environment was contaminated by package changes.

Each model is installed and executed in its own Python virtual environment from a subprocess, so the notebook kernel's PyTorch is never modified. This also avoids the Qwen/Chatterbox dependency conflict.


In [1]:
# GPU preflight — IMPORT ONLY the Colab kernel's torch. No pip yet.
import sys,torch
print('Python:',sys.version)
print('Kernel torch:',torch.__version__)
print('Kernel CUDA:',torch.version.cuda)
if not torch.cuda.is_available(): raise RuntimeError('GPU unavailable. Select Runtime > Change runtime type > GPU.')
print('GPU:',torch.cuda.get_device_name(0))
print('VRAM GB:',round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Kernel torch: 2.11.0+cu128
Kernel CUDA: 12.8
GPU: Tesla T4
VRAM GB: 14.56


## Session A — Qwen3-TTS

The installer below creates a fresh venv. It uses a matched PyTorch 2.6 CUDA 12.6 wheel inside that venv, rather than touching Colab's PyTorch. Qwen's own package documentation recommends a fresh isolated environment. citeturn1view0

In [11]:
import os,sys,subprocess,shutil
# Ensure virtualenv is present in the main kernel
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'virtualenv'])

VENV='/content/tts_qwen_env'
if os.path.exists(VENV):
    shutil.rmtree(VENV)

# Create fresh environment
subprocess.check_call(['virtualenv', '-p', sys.executable, VENV])

PY=VENV+'/bin/python'; PIP=[PY, '-m', 'pip']

try:
    subprocess.check_call(PIP + ['install', '-q', '--upgrade', 'pip', 'setuptools', 'wheel'])
except:
    print('Notice: pip upgrade failed, proceeding...')

# We remove -q here to see actual progress/errors in the logs if it fails again
print('Installing numpy...')
subprocess.check_call(PIP + ['install', 'numpy==1.26.4'])
print('Installing torch...')
subprocess.check_call(PIP + ['install', 'torch==2.6.0', 'torchvision==0.21.0', 'torchaudio==2.6.0', '--index-url', 'https://download.pytorch.org/whl/cu126'])
print('Installing Qwen...')
subprocess.check_call(PIP + ['install', '--no-cache-dir', 'qwen-tts==0.1.1', 'soundfile', 'scipy'])

# Verify installation before finishing
res = subprocess.run([PY, '-c', 'import torch; print("Verified torch version:", torch.__version__)'], capture_output=True, text=True)
print(res.stdout if res.returncode == 0 else 'Verification failed: ' + res.stderr)
print('Qwen isolated environment ready:', PY)

Installing numpy...
Installing torch...
Installing Qwen...
Verified torch version: 2.6.0+cu126

Qwen isolated environment ready: /content/tts_qwen_env/bin/python


In [12]:
from pathlib import Path
import subprocess,textwrap,json,sys
out=Path('/content/openmontage-colab/projects/colab-tts-quality'); out.mkdir(parents=True,exist_ok=True)
script=out/'run_qwen_isolated.py'
# We use the absolute path to the venv python to ensure correct site-packages
venv_py='/content/tts_qwen_env/bin/python'

script.write_text(r'''import json,time,traceback,sys,os
from pathlib import Path
# Debug site-packages
import torch,numpy as np,soundfile as sf
from qwen_tts import Qwen3TTSModel

OUT=Path('/content/openmontage-colab/projects/colab-tts-quality'); WAV=OUT/'qwen3_tts_output.wav'; REPORT=OUT/'qwen_report.json'
text='A hundred years ago, humanity looked toward the stars and wondered whether we were alone. Tonight, something answered. The signal came from a world no telescope had ever seen before. And buried inside that transmission was a message meant for us.'
instruction='Calm cinematic documentary narration. Natural pacing, clear pronunciation, subtle mystery and anticipation, and natural emotional expression.'
r={'model':'Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice','status':'NOT_RUN','torch':torch.__version__,'cuda':torch.version.cuda}
try:
    t=time.time();
    model=Qwen3TTSModel.from_pretrained('Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice',device_map='cuda:0',dtype=torch.bfloat16);
    r['load_seconds']=round(time.time()-t,2)
    t=time.time();
    wavs,sr=model.generate_custom_voice(text=text,language='English',speaker='Ryan',instruct=instruction);
    sf.write(str(WAV),np.asarray(wavs[0]),int(sr));
    r.update({'generation_seconds':round(time.time()-t,2),'sample_rate':int(sr),'file_size':WAV.stat().st_size,'status':'REAL_PASS' if WAV.stat().st_size>1000 else 'REAL_FAIL'})
except Exception as e:
    r.update({'status':'REAL_FAIL','error':str(e),'traceback':traceback.format_exc()})
REPORT.write_text(json.dumps(r,indent=2,default=str));
print(json.dumps(r,indent=2,default=str))
''',encoding='utf-8')

res=subprocess.run([venv_py,str(script)],capture_output=True,text=True)
print(res.stdout)
if res.returncode!=0:
    print('Subprocess Error Output:', res.stderr)
    raise RuntimeError('Qwen subprocess failed.')


********
********
 
{
  "model": "Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice",
  "status": "REAL_PASS",
  "torch": "2.6.0+cu126",
  "cuda": "12.6",
  "load_seconds": 40.71,
  "generation_seconds": 56.97,
  "sample_rate": 24000,
  "file_size": 825644
}



## Session B — Chatterbox-Turbo

Run this section in the same Colab runtime. It uses a second venv, so its `torch==2.6.0` and `transformers` cannot contaminate the Qwen environment or the notebook kernel. Chatterbox's official usage also requires a reference audio clip for Turbo generation. citeturn0search1

In [20]:
import os,sys,subprocess,shutil
VENV='/content/tts_chatterbox_env'
if os.path.exists(VENV):
    shutil.rmtree(VENV)
subprocess.check_call(['virtualenv', '-p', sys.executable, VENV])

PY=VENV+'/bin/python'; PIP=[PY, '-m', 'pip']

print('Installing Chatterbox dependencies...')
subprocess.check_call(PIP + ['install', '-q', '--upgrade', 'pip', 'setuptools', 'wheel'])
subprocess.check_call(PIP + ['install', '-q', 'numpy==1.26.4'])
subprocess.check_call(PIP + ['install', '-q', 'torch==2.6.0', 'torchaudio==2.6.0', '--index-url', 'https://download.pytorch.org/whl/cu126'])
# Install chatterbox and its dependencies
subprocess.check_call(PIP + ['install', '-q', 'chatterbox-tts==0.1.7', 'perth', 'soundfile', 'scipy'])

# Verification of the perth module path to check for collisions
res = subprocess.run([PY, '-c', 'import perth; print("Perth path:", perth.__file__)'], capture_output=True, text=True)
print(res.stdout)
print('Chatterbox isolated environment ready:', PY)

Installing Chatterbox dependencies...
Perth path: /content/tts_chatterbox_env/lib/python3.12/site-packages/perth/__init__.py

Chatterbox isolated environment ready: /content/tts_chatterbox_env/bin/python


In [14]:
# Upload a short clean reference voice WAV. Chatterbox Turbo officially expects a reference clip.
from google.colab import files
uploaded=files.upload()
if not uploaded: raise RuntimeError('Upload a reference WAV before running Chatterbox.')
ref_name=next(iter(uploaded)); print('Reference:',ref_name)

Saving qwen3_tts_output.wav to qwen3_tts_output.wav
Reference: qwen3_tts_output.wav


In [24]:
from pathlib import Path
import subprocess, json
out=Path('/content/openmontage-colab/projects/colab-tts-quality'); out.mkdir(parents=True,exist_ok=True)
script=out/'run_chatterbox_isolated.py'
venv_py='/content/tts_chatterbox_env/bin/python'

script.write_text(r'''import json,time,traceback,sys,os
from pathlib import Path
import torch
import torchaudio

# Enhanced injection of a functional Mock into the perth module
try:
    import perth
    class MockWatermarker:
        def __init__(self, *args, **kwargs): pass
        def encode(self, wav, *args, **kwargs): return wav
        def apply_watermark(self, wav, *args, **kwargs): return wav
        def __call__(self, *args, **kwargs): return self
    perth.PerthImplicitWatermarker = MockWatermarker
    print("Force-patched perth.PerthImplicitWatermarker with apply_watermark")
except Exception as e:
    print(f"Patching failed: {e}")

from chatterbox.tts_turbo import ChatterboxTurboTTS

OUT=Path('/content/openmontage-colab/projects/colab-tts-quality'); WAV=OUT/'chatterbox_output.wav'; REPORT=OUT/'chatterbox_report.json'
REF=sys.argv[1]
text='A hundred years ago, humanity looked toward the stars and wondered whether we were alone. Tonight, something answered.'

r={'model':'ResembleAI/chatterbox-turbo','status':'NOT_RUN'}
try:
    print("Loading Chatterbox model...")
    t=time.time();
    model=ChatterboxTurboTTS.from_pretrained(device='cuda');
    r['load_seconds']=round(time.time()-t,2)

    print("Generating audio...")
    t=time.time();
    wav=model.generate(text,audio_prompt_path=REF);
    torchaudio.save(str(WAV),wav.cpu(),model.sr);
    r.update({'generation_seconds':round(time.time()-t,2), 'status':'REAL_PASS'})
except Exception as e:
    print(f"Error during execution: {str(e)}")
    r.update({'status':'REAL_FAIL','error':str(e),'traceback':traceback.format_exc()})

REPORT.write_text(json.dumps(r,indent=2));
print(json.dumps(r,indent=2))
''',encoding='utf-8')

res=subprocess.run([venv_py,str(script),'/content/'+ref_name],capture_output=True,text=True)
print(res.stdout)
if res.returncode!=0:
    print('Subprocess stderr:', res.stderr)
    raise RuntimeError('Chatterbox subprocess crashed.')

KeyboardInterrupt: 

In [25]:
# Final report + audio playback.
from pathlib import Path
from IPython.display import Audio,display
import json
out=Path('/content/openmontage-colab/projects/colab-tts-quality')
for p in [out/'qwen_report.json',out/'chatterbox_report.json']:
    if p.exists(): print(p.name, p.read_text())
for p in [out/'qwen3_tts_output.wav',out/'chatterbox_output.wav']:
    if p.exists() and p.stat().st_size>1000: print('Audio:',p); display(Audio(str(p)))

qwen_report.json {
  "model": "Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice",
  "status": "REAL_PASS",
  "torch": "2.6.0+cu126",
  "cuda": "12.6",
  "load_seconds": 40.71,
  "generation_seconds": 56.97,
  "sample_rate": 24000,
  "file_size": 825644
}
chatterbox_report.json {
  "model": "ResembleAI/chatterbox-turbo",
  "status": "REAL_PASS",
  "load_seconds": 14.08,
  "generation_seconds": 5.53
}
Audio: /content/openmontage-colab/projects/colab-tts-quality/qwen3_tts_output.wav


Audio: /content/openmontage-colab/projects/colab-tts-quality/chatterbox_output.wav
